<a href="https://colab.research.google.com/github/Ekansh0801/Movie-Recommendation-system/blob/main/MovieRecommendationSystem_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation System (TF-IDF + Cosine Similarity)
Content-based recommender on the TMDB 5000 dataset.

In [9]:
import ast
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load and merge

In [10]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')
print(movies.shape, credits.shape)

(4803, 20) (4803, 4)


**Why merge on the ID, not the title:** some titles appear twice (e.g. two different films called *Batman*).
Merging on `title` cross-matches them and creates extra rows with the wrong cast/plot
(this is why the old version had 4806 rows instead of 4803).

In [11]:
print("Duplicate titles:", movies['title'][movies['title'].duplicated()].tolist())

credits = credits.drop(columns=['title'])
movies = movies.merge(credits, left_on='id', right_on='movie_id')
print(movies.shape)   # should stay 4803

Duplicate titles: ['The Host', 'Out of the Blue', 'Batman']
(4803, 23)


## 2. Keep content columns and clean

In [12]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
print(movies.isnull().sum())
movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)
movies.shape

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64


(4800, 7)

## 3. Feature engineering

In [13]:
def convert(text):
    # '[{"id": 28, "name": "Action"}, ...]'  ->  ['Action', ...]
    return [i['name'] for i in ast.literal_eval(text)]

def convert3(text):
    # top 3 cast members only
    return [i['name'] for i in ast.literal_eval(text)[:3]]

def fetch_director(text):
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

def collapse(L):
    # 'Sam Worthington' -> 'SamWorthington' so each person is ONE token
    return [i.replace(" ", "") for i in L]

movies['genres'] = movies['genres'].apply(convert).apply(collapse)
movies['keywords'] = movies['keywords'].apply(convert).apply(collapse)
movies['cast'] = movies['cast'].apply(convert3).apply(collapse)
movies['crew'] = movies['crew'].apply(fetch_director).apply(collapse)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

new = movies[['movie_id', 'title', 'tags']].copy()
new['tags'] = new['tags'].apply(lambda x: " ".join(x).lower())
new.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


## 4. TF-IDF vectorization
TF-IDF down-weights words that appear in many movies (e.g. *action*, *film*) and boosts distinctive ones.
The result stays a **sparse** matrix: no `.toarray()`, which would waste ~190 MB.

In [14]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vector = tfidf.fit_transform(new['tags'])
print(vector.shape, f"non-zero entries: {vector.nnz / (vector.shape[0] * vector.shape[1]):.2%}")

(4800, 5000) non-zero entries: 0.56%


## 5. Cosine similarity

In [15]:
similarity = cosine_similarity(vector).astype(np.float32)   # float32 halves the file size
print(similarity.shape, f"{similarity.nbytes / 1e6:.0f} MB")

(4800, 4800) 92 MB


## 6. Recommend

In [16]:
def recommend(movie, n=5):
    matches = new.index[new['title'].str.lower() == movie.lower()]
    if len(matches) == 0:
        print(f"'{movie}' not found")
        return
    index = matches[0]
    distances = sorted(enumerate(similarity[index]), reverse=True, key=lambda x: x[1])
    for i, score in distances[1:n + 1]:      # skip [0]: the movie itself
        print(f"{new.iloc[i].title}  ({score:.2f})")

recommend('Batman')

Batman & Robin  (0.44)
Batman Returns  (0.30)
The Dark Knight Rises  (0.28)
Batman Begins  (0.26)
Batman v Superman: Dawn of Justice  (0.22)


## 7. Comparison: bag-of-words vs TF-IDF
Same tags, only the vectorizer changes. Write down what differs; this is your interview story.

In [17]:
bow = CountVectorizer(max_features=5000, stop_words='english')
bow_vector = bow.fit_transform(new['tags'])

def top_titles(vec, movie, n=5):
    i = new.index[new['title'] == movie][0]
    s = cosine_similarity(vec[i], vec).ravel()
    s[i] = -1
    return [new.iloc[j].title for j in np.argsort(-s)[:n]]

for m in ['Avatar', 'The Dark Knight Rises', 'Toy Story', 'Titanic', 'The Avengers']:
    if m in new['title'].values:
        print(f"\n=== {m} ===")
        print("Bag-of-words:", top_titles(bow_vector, m))
        print("TF-IDF      :", top_titles(vector, m))


=== Avatar ===
Bag-of-words: ['Titan A.E.', 'Small Soldiers', "Ender's Game", 'Independence Day', 'Aliens vs Predator: Requiem']
TF-IDF      : ['Aliens', 'Falcon Rising', 'Battle: Los Angeles', 'Apollo 18', 'Star Trek Into Darkness']

=== The Dark Knight Rises ===
Bag-of-words: ['The Dark Knight', 'Batman Begins', 'Batman Returns', 'Batman Forever', 'Batman']
TF-IDF      : ['The Dark Knight', 'Batman Returns', 'Batman Begins', 'Batman Forever', 'Batman']

=== Toy Story ===
Bag-of-words: ['Toy Story 2', 'Toy Story 3', 'The 40 Year Old Virgin', 'Heartbeeps', "Max Keeble's Big Move"]
TF-IDF      : ['Toy Story 3', 'Toy Story 2', 'The 40 Year Old Virgin', 'Factory Girl', 'Class of 1984']

=== Titanic ===
Bag-of-words: ['Captain Phillips', 'The Notebook', 'In the Heart of the Sea', 'Ghost Ship', 'Poseidon']
TF-IDF      : ['Ghost Ship', 'Poseidon', 'In the Heart of the Sea', 'Pirates of the Caribbean: On Stranger Tides', 'Triangle']

=== The Avengers ===
Bag-of-words: ['Avengers: Age of Ultr

## 8. Save for the Streamlit app

In [18]:
pickle.dump(new, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))   # ~92 MB, under GitHub's 100 MB limit